# FINS 实验流水线

一条 notebook 串起 4 个模块（原 notebook 已 py 化）：

| 步骤       | 模块 | 作用                                                                                           |
|------------|---|------------------------------------------------------------------------------------------------|
| 1 生成配置 | `_load.py` | 按拓扑 / 时序 / 负载随机生成 pipeline cfg → `pipeline/*.json`                             |
| 2 采集     | `_test.py` | 【fins 测试专用】每份 cfg 起 `bin/client` + `bin/server` 跑 `dur_s` 秒 → trace 复制到 `result/`              |
| 3 标准化   | `std.py` | 【fins 测试专用】 用 JSON 的执行用时在 `execute→complete` 内插 `working` 行 → `result_std/` |
| 4 分析画图 | `plot.py` | 核心甘特、核利用率、生命周期分布                                                               |

> 三个数据目录都在**仓库根**：`pipeline/`（配置）、`result/`（原始 trace）、`result_std/`（标准化 trace）。
> 下面所有相对路径都按仓库根解析，不依赖 notebook 的 cwd。

In [1]:
import importlib

# 导入你的基础模块与工具库
import _load
import _test
import _lttng2csv
# import _csv2graph

# 导入绘图模块
import _csv2overhead
import _csv2timeline
from _csv2overhead import analyze_cpu_utilization
from _csv2timeline import generate_execution_gantt

# 将所有需要支持热重载的本地模块统一放到元组中刷新
for m in (_load, _test, _csv2timeline, _csv2overhead):
  importlib.reload(m)

## 1. 生成配置（`_load.py`）

5 种拓扑（multihop / fork / join / feedback / mixed，均可混入 acc 的 hist 窗口读）
+ 时序（`ptimed`、`period_divisors`）+ 负载（`u`、`H_ms` 等）随机生成，
文件名 `<kind>_u<u>_m<m>_ms<桶>_s<seed>.json`。

In [3]:
my_cfg_dir = "tool/pipeline"
CONFIG = {

    "topology": {

        "multihop": {
            "paths": (1, 4),
            "depth": (2, 8),
        },

        "fork": {
            "fan": (2, 8),
            "bdepth": (1, 3),
        },

        "join": {
            "fan": (2, 8),
            "bdepth": (1, 3),
            "tail": (1, 3),
        },

        "feedback": {
            "depth": (3, 8),
            "histN": (3, 8),
        },

        "mixed": {
            "nseg": (3, 8),

            "chain_prob": 0.45,
            "fork_join_prob": 0.30,
            "feedback_prob": 0.25,
        },
    },

    "temporal": {

        # Probability that a non-source node is timed.
        "ptimed": 0.35,

        # T values are H / divisor.
        #
        # Example:
        #
        # H=100
        #
        # divisor 1 -> 100 ms
        # divisor 2 -> 50 ms
        # divisor 4 -> 25 ms
        #
        "period_divisors": [1, 2, 4, 5, 10, 20],
    },

    "workload": {

        "u": [
            0.1,
            0.3,
            0.5,
            0.7,
            0.9,
        ],

        "workers": [
            1,
            2,
            3,
        ],

        # Hyperperiod.
        "H_ms": 100,

        # ----------------------------------------------------
        # Makespan classification.
        #
        # Example H=100, width=5:
        #
        # bin 00: [0,5)
        # bin 01: [5,10)
        # ...
        # bin 18: [90,95)
        # bin 19: [95,100]
        #
        # The last bucket includes H.
        # ----------------------------------------------------

        "makespan_bin_width_ms": 10.0,

        # Anything above this goes into overflow.
        #
        # None:
        #     use H_ms.
        #
        "makespan_max_ms": 80,

        # Number of final samples per bucket.
        "n_per": 2,

        # ----------------------------------------------------
        # Generation policy.
        # ----------------------------------------------------

        # Initial candidate generation.
        "initial_attempts": 1000,

        # Extra attempts for incomplete buckets.
        "refill_attempts": 1000,

        # Maximum number of refill rounds.
        "max_refill_rounds": 1,

        # Keep at most:
        #
        #     n_per * candidate_factor
        #
        # candidates per bucket.
        #
        "candidate_factor": 1,

        # Random seed.
        "seed_base": 20260809,

    },

    "solver": {

        # Utilization numerical tolerance.
        "u_tolerance": 1e-8,

        # Minimum WCET.
        #
        # Plugin cfg is in microseconds.
        #
        "min_wcet_us": 1000,

        # C_i <= T_i * max_c_ratio
        "max_c_ratio": 0.2,

        # Number of attempts used to find a C allocation.
        "c_attempts": 300,
    },
}

_load.generate_all(
    out_dir=my_cfg_dir,
    config=CONFIG
)


[CONFIG] multihop u=0.1 m=1
[PHASE-1] kind=multihop u=0.1 m=1 attempts=1000
[PHASE-1] generated=8
[REFILL-1] missing=4 kind(s)
[REFILL-1] generated=0
[INCOMPLETE] kind=multihop u=0.1 m=1
  bucket=02 [20.0,30.0] available=0 required=2
  bucket=03 [30.0,40.0] available=0 required=2
  bucket=04 [40.0,50.0] available=0 required=2
  bucket=06 [60.0,70.0] available=0 required=2
[SAVED] 8 files

[CONFIG] multihop u=0.1 m=2
[PHASE-1] kind=multihop u=0.1 m=2 attempts=1000
[PHASE-1] generated=8
[REFILL-1] missing=4 kind(s)
[REFILL-1] generated=1
[INCOMPLETE] kind=multihop u=0.1 m=2
  bucket=00 [0.0,10.0] available=1 required=2
  bucket=03 [30.0,40.0] available=0 required=2
  bucket=04 [40.0,50.0] available=0 required=2
  bucket=06 [60.0,70.0] available=0 required=2
[SAVED] 8 files

[CONFIG] multihop u=0.1 m=3
[PHASE-1] kind=multihop u=0.1 m=3 attempts=1000
[PHASE-1] generated=9
[REFILL-1] missing=4 kind(s)
[REFILL-1] generated=3
[INCOMPLETE] kind=multihop u=0.1 m=3
  bucket=00 [0.0,10.0] availa

['/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms00_s20349364.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms00_s20349363.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms01_s20349389.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms01_s20349448.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms05_s20349928.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms05_s20349361.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms07_s20349931.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m1_ms07_s20349814.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m2_ms01_s20725514.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m2_ms01_s20725531.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline/multihop_u10_m2_ms02_s20725597.json',
 '/home/jenny/Documents/GitHub/f

## 2. 采集（`_test.py`）

对每份 cfg：起 `client` + 发配置 + 跑 `dur_s` 秒 → 终止 → 把 `tool/temp/tracing.csv`
复制成 `result/<cfg名>.csv`（原件保留，只搬原始 trace，不算指标）。

正式实验要走独占核：`cores="1-6"` 会改用 `sudo tool/client.sh <cores> <workers>`（需要 root）。

In [4]:
# 指定你的目录
my_cfg_dir = "tool/pipeline"
my_result_dir = "tool/result"

# 一键运行（cpu_offset=1 代表核心从 1 开始排，如果 m=3 就会自动分配核 "1-3"）
_test.run_all(
    target_cfg_dir=my_cfg_dir,
    target_result_dir=my_result_dir,
    run_time=10.0,     # 运行时间
    cpu_offset=1      # 起始核号（避开核心 0 给系统）
)

🔒 该评测脚本需要 root 权限来配置 Cgroup 与绑定核心
🚀 开始智能批量评测（自动从文件名匹配 m）
   📂 输入配置目录: tool/pipeline
   📁 结果输出目录: tool/result
   📊 发现测试用例: 26 个

进度 [1/26]: feedback_u30_m1_ms05_s20277872.json

>>> [开始测试] feedback_u30_m1_ms05_s20277872 (自动解析: m=1 -> workers=1, cores=1)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_feedback_u30_m1_ms05_s20277872_144953)...
  [2/5] 等待 LTTng 预热 1.0s 并推送配置...
  [2/5] 启动 Client 进程 (裸跑模式, workers=1)...
  [3/5] 等待客户端预热 1.0s 并推送配置...
  [Server] 配置灌入成功，持续运行 10.0s...
  [4/5] 停止并销毁 LTTng 追踪会话...
  [5/5] 关闭清理 Client 进程 (pgid=92899)...
  [成功] 轨迹数据已归档至: /home/jenny/Documents/GitHub/fins/tool/result/feedback_u30_m1_ms05_s20277872_144953/trace
  [验证] ✅ 成功捕捉到 11239 条目标跟踪事件！

进度 [2/26]: feedback_u30_m2_ms05_s20581484.json

>>> [开始测试] feedback_u30_m2_ms05_s20581484 (自动解析: m=2 -> workers=2, cores=1-2)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_feedback_u30_m2_ms05_s20581484_145009)...
  [2/5] 等待 LTTng 预热 1.0s 并推送配置...
  [2/5] 启动 Client 进程 (裸跑模式, workers=2)...
  [3/5] 等待客户端预热 1.0s 并推送配置...
  [Server] 配置灌入成功

# 3. 转义（tran）
1. lttng 2 timeline
2. lttng 2 overhead

In [5]:
_lttng2csv.run_export(
    results_dir="./result",
    outdir="./result",
    after_us=2000 * 1000,
    preempt_all=True,
    cpu_start=1,
)


开始处理 26 个实验 -> ./result/  输出: timeline, preempt
  [成功] feedback_u30_m1_ms05_s20277872_144953  t0=22091884307774  cpus=1-1(m=1)  ->  feedback_u30_m1_ms05_s20277872_144953_timeline.csv (9021 行) | feedback_u30_m1_ms05_s20277872_144953_preempt.csv (11698 行)
  [成功] feedback_u30_m2_ms05_s20581484_145009  t0=22107877504609  cpus=1-2(m=2)  ->  feedback_u30_m2_ms05_s20581484_145009_timeline.csv (6787 行) | feedback_u30_m2_ms05_s20581484_145009_preempt.csv (24381 行)
  [成功] feedback_u30_m3_ms05_s20579977_145025  t0=22123870047474  cpus=1-3(m=3)  ->  feedback_u30_m3_ms05_s20579977_145025_timeline.csv (11460 行) | feedback_u30_m3_ms05_s20579977_145025_preempt.csv (33115 行)
  [成功] feedback_u50_m1_ms05_s20284954_145041  t0=22139858384589  cpus=1-1(m=1)  ->  feedback_u50_m1_ms05_s20284954_145041_timeline.csv (6203 行) | feedback_u50_m1_ms05_s20284954_145041_preempt.csv (8715 行)
  [成功] feedback_u50_m2_ms05_s21235484_145057  t0=22155773643136  cpus=1-2(m=2)  ->  feedback_u50_m2_ms05_s21235484_145057_timeli

## 4. 分析与画图（`plot.py`）



In [ ]:
# CIE_FIFO_IPC_nuc12-noturbo_hythread
# feedback_u70_m3_ms05_s20632672_195219_preempt.csv
# feedback_u70_m3_ms05_s20632672_195219_timeline.csv
fig_gantt_ROS_7B = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_timeline.csv",
    zoom_window_ms=[0, 1000])

# CIE_FIFO_IPC_nuc12_1kB
# feedback_u70_m3_ms05_s20632672_152109_preempt.csv
# feedback_u70_m3_ms05_s20632672_152109_timeline.csv
fig_gantt_ROS_1kB = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_timeline.csv",
    zoom_window_ms=[0, 1000])

# CIE_FIFO_IPC_nuc12_1MB
# feedback_u70_m3_ms05_s20632672_152938_preempt.csv
# feedback_u70_m3_ms05_s20632672_152938_timeline.csv
# fig_gantt_ROS_1MB = generate_execution_gantt(
#     preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_152938_preempt.csv",
#     timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_152938_timeline.csv",
#     zoom_window_ms=[0, 1000])

# FINS_7B
# feedback_u70_m3_ms05_s20632672_210928_preempt.csv
# feedback_u70_m3_ms05_s20632672_210928_timeline.csv
fig_gantt_FINS_7B = generate_execution_gantt(
    preempt_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_preempt.csv",
    timeline_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_timeline.csv",
    zoom_window_ms=[0, 1000])

# FINS_1kB
# feedback_u70_m3_ms05_s20632672_211315_preempt.csv
# feedback_u70_m3_ms05_s20632672_211315_timeline.csv
fig_gantt_FINS_1kB = generate_execution_gantt(
    preempt_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_preempt.csv",
    timeline_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_timeline.csv",
    zoom_window_ms=[0, 1000])

# FINS_1MB
# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
fig_gantt_FINS_1MB = generate_execution_gantt(
    preempt_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
    timeline_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_ROS_7B.show()
fig_gantt_ROS_1kB.show()
# fig_gantt_ROS_1MB.show()
fig_gantt_FINS_7B.show()
fig_gantt_FINS_1kB.show()
fig_gantt_FINS_1MB.show()

In [ ]:
# CIE_FIFO_IPC_nuc12-noturbo_hythread
# feedback_u70_m3_ms05_s20632672_195219_preempt.csv
# feedback_u70_m3_ms05_s20632672_195219_timeline.csv
fig_util_ROS_7B = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_7B/feedback_u70_m3_ms05_s20632672_195219_timeline.csv",
    analysis_window_ms=[50, 5000])

# CIE_FIFO_IPC_nuc12_1kB
# feedback_u70_m3_ms05_s20632672_152109_preempt.csv
# feedback_u70_m3_ms05_s20632672_152109_timeline.csv
fig_util_ROS_1kB = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1kB/feedback_u70_m3_ms05_s20632672_152109_timeline.csv",
    analysis_window_ms=[50, 5000])

# CIE_FIFO_IPC_nuc12_1MB
# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
# fig_util_ROS_1MB = analyze_cpu_utilization(
#     preempt_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
#     timeline_csv = "typical/CIE_FIFO_IPC_nuc12_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
#     analysis_window_ms=[0, 1000])

# FINS
# feedback_u70_m3_ms05_s20632672_210928_preempt.csv
# feedback_u70_m3_ms05_s20632672_210928_timeline.csv
fig_util_FINS_7B = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_preempt.csv",
    timeline_csv = "typical/FINS_7B/feedback_u70_m3_ms05_s20632672_210928_timeline.csv",
    analysis_window_ms=[50, 5000])

# feedback_u70_m3_ms05_s20632672_211315_preempt.csv
# feedback_u70_m3_ms05_s20632672_211315_timeline.csv
fig_util_FINS_1kB = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_preempt.csv",
    timeline_csv = "typical/FINS_1kB/feedback_u70_m3_ms05_s20632672_211315_timeline.csv",
    analysis_window_ms=[50, 5000])

# feedback_u70_m3_ms05_s20632672_213624_preempt.csv
# feedback_u70_m3_ms05_s20632672_213624_timeline.csv
fig_util_FINS_1MB = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
    timeline_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
    analysis_window_ms=[50, 5000])

# multihop_u10_m1_ms05_s20743227_231518_preempt.csv
# multihop_u10_m1_ms05_s20743227_231518_timeline.csv
fig_util_FINS_1MB = analyze_cpu_utilization(
    preempt_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_preempt.csv",
    timeline_csv = "typical/FINS_1MB/feedback_u70_m3_ms05_s20632672_213624_timeline.csv",
    analysis_window_ms=[50, 5000])

fig_util_ROS_7B.show()
fig_util_ROS_1kB.show()
# fig_util_ROS_1MB.show()
fig_util_FINS_7B.show()
fig_util_FINS_1kB.show()
fig_util_FINS_1MB.show()

In [8]:
# feedback_u30_m1_ms05_s20277872_144953_preempt.csv
# feedback_u30_m1_ms05_s20277872_144953_timeline.csv
# feedback_u30_m2_ms05_s20581484_145009_preempt.csv
# feedback_u30_m2_ms05_s20581484_145009_timeline.csv
# feedback_u30_m3_ms05_s20579977_145025_preempt.csv
# feedback_u30_m3_ms05_s20579977_145025_timeline.csv
# feedback_u50_m1_ms05_s20284954_145041_preempt.csv
# feedback_u50_m1_ms05_s20284954_145041_timeline.csv
# feedback_u50_m2_ms05_s21235484_145057_preempt.csv
# feedback_u50_m2_ms05_s21235484_145057_timeline.csv
# feedback_u70_m1_ms05_s20512128_145114_preempt.csv
# feedback_u70_m1_ms05_s20512128_145114_timeline.csv
# feedback_u90_m1_ms05_s21254923_145130_preempt.csv
# feedback_u90_m1_ms05_s21254923_145130_timeline.csv
# fork_u30_m1_ms05_s20518563_145147_preempt.csv
# fork_u30_m1_ms05_s20518563_145147_timeline.csv
# fork_u30_m2_ms05_s20516860_145202_preempt.csv
# fork_u30_m2_ms05_s20516860_145202_timeline.csv
# fork_u30_m3_ms05_s20820517_145218_preempt.csv
# fork_u30_m3_ms05_s20820517_145218_timeline.csv
# fork_u50_m3_ms05_s20870345_145233_preempt.csv
# fork_u50_m3_ms05_s20870345_145233_timeline.csv
# fork_u70_m1_ms05_s20801687_145249_preempt.csv
# fork_u70_m1_ms05_s20801687_145249_timeline.csv
# fork_u90_m1_ms05_s21058989_145305_preempt.csv
# fork_u90_m1_ms05_s21058989_145305_timeline.csv
# join_u30_m1_ms05_s20931847_145321_preempt.csv
# join_u30_m1_ms05_s20931847_145321_timeline.csv
# join_u30_m2_ms05_s21235524_145337_preempt.csv
# join_u30_m2_ms05_s21235524_145337_timeline.csv
# join_u30_m3_ms05_s30865459_145353_preempt.csv
# join_u30_m3_ms05_s30865459_145353_timeline.csv
# join_u50_m1_ms05_s20454978_145409_preempt.csv
# join_u50_m1_ms05_s20454978_145409_timeline.csv
# join_u50_m2_ms05_s20456264_145425_preempt.csv
# join_u50_m2_ms05_s20456264_145425_timeline.csv
# join_u50_m3_ms07_s31064752_145441_preempt.csv
# join_u50_m3_ms07_s31064752_145441_timeline.csv
# multihop_u30_m2_ms04_s21084021_145457_preempt.csv
# multihop_u30_m2_ms04_s21084021_145457_timeline.csv
# multihop_u30_m2_ms05_s21083838_145513_preempt.csv
# multihop_u30_m2_ms05_s21083838_145513_timeline.csv
# multihop_u30_m2_ms05_s21083869_145530_preempt.csv
# multihop_u30_m2_ms05_s21083869_145530_timeline.csv
# multihop_u50_m2_ms04_s21125652_145546_preempt.csv
# multihop_u50_m2_ms04_s21125652_145546_timeline.csv
# multihop_u90_m1_ms04_s20541899_145602_preempt.csv
# multihop_u90_m1_ms04_s20541899_145602_timeline.csv
# multihop_u90_m1_ms07_s20542115_145618_preempt.csv
# multihop_u90_m1_ms07_s20542115_145618_timeline.csv
# multihop_u90_m2_ms07_s20912328_145635_preempt.csv
# multihop_u90_m2_ms07_s20912328_145635_timeline.csv
fig_gantt_FINS = generate_execution_gantt(
    preempt_csv = "result/join_u50_m3_ms07_s31064752_145441_preempt.csv",
    timeline_csv = "result/join_u50_m3_ms07_s31064752_145441_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_util_FINS = analyze_cpu_utilization(
    preempt_csv = "result/join_u50_m3_ms07_s31064752_145441_preempt.csv",
    timeline_csv = "result/join_u50_m3_ms07_s31064752_145441_timeline.csv",
    analysis_window_ms=[50, 5000])

fig_gantt_FINS.show()
fig_util_FINS.show()

正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
  worker 总 CPU 时间 : 15351.354 ms
    - 任务    : 15245.732 ms (99.3%)
    - Overhead: 105.623 ms (0.7%)
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html

逐核利用率：
   CPU 2  Active  55.06%  Overhead   0.35%  Idle  44.59%   (total 4950.0 ms)
   CPU 1  Active  52.51%  Overhead   0.42%  Idle  47.07%   (total 4950.0 ms)
   CPU 3  Active  45.38%  Overhead   0.33%  Idle  54.30%   (total 4950.0 ms)

全局加权：
  Active    50.98%  (7570.6 ms / 14850.0 ms)
  Overhead   0.37%  (54.3 ms / 14850.0 ms)
  Idle      48.65%  (7225.1 ms / 14850.0 ms)
